# Modelagem e Catálogo de Dados

Objetivo: Organizar os dados em um modelo dimensional do tipo Esquema Estrela.

In [0]:
CATALOG = "workspace"
from pyspark.sql import functions as F
from pyspark.sql import Window

> Criação da dimensão de títulos

Armazena as principais características de filmes, séries e episódios, incluindo tipo de produção, títulos, anos de lançamento e duração.

In [0]:
silver_title_rated = spark.table(f"{CATALOG}.silver.title_rated")

dim_title = (
    silver_title_rated
    .select(
        F.col("tconst").alias("title_id"),
        F.col("title_type_pt").alias("title_type"),   # Filme / Série / Episódio
        F.col("titleType").alias("title_type_original"),
        F.col("primaryTitle").alias("primary_title"),
        F.col("originalTitle").alias("original_title"),
        F.col("startYear").alias("start_year"),
        F.col("endYear").alias("end_year"),
        F.col("runtimeMinutes").alias("runtime_minutes"),
        F.col("isAdult").alias("is_adult"),
    )
)

dim_title.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.gold.dim_title")
print(f"gold.dim_title: {spark.table(f'{CATALOG}.gold.dim_title').count():,} linhas")

gold.dim_title: 1,432,172 linhas


> Criação da dimensão de gêneros

Cria uma dimensão de gêneros e uma tabela de relacionamento que permite associar cada título a um ou mais gêneros.

In [0]:
exploded_genres = (
    silver_title_rated
    .select(F.col("tconst").alias("title_id"), F.explode_outer("genres_array").alias("genre_name"))
    .filter(F.col("genre_name").isNotNull())
)

dim_genre = (
    exploded_genres
    .select("genre_name")
    .distinct()
    .withColumn("genre_id", F.row_number().over(Window.orderBy("genre_name")))
    .select("genre_id", "genre_name")
)

dim_genre.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.gold.dim_genre")

bridge_title_genre = (
    exploded_genres
    .join(dim_genre, on="genre_name", how="inner")
    .select("title_id", "genre_id")
)

bridge_title_genre.write.mode("overwrite").format("delta").saveAsTable(
    f"{CATALOG}.gold.bridge_title_genre"
)

print(f"gold.dim_genre: {spark.table(f'{CATALOG}.gold.dim_genre').count():,} linhas")
print(f"gold.bridge_title_genre: {spark.table(f'{CATALOG}.gold.bridge_title_genre').count():,} linhas")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


gold.dim_genre: 29 linhas
gold.bridge_title_genre: 2,871,300 linhas


> Criação da dimensão de elenco

Armazena os atores e atrizes associados aos títulos analisados, contendo identificador do profissional, nome e ano de nascimento.

In [0]:
silver_cast_people = spark.table(
    f"{CATALOG}.silver.cast_people"
)

dim_person = (
    silver_cast_people
    .select(
        F.col("nconst").alias("person_id"),
        F.col("primaryName").alias("primary_name"),
        F.col("birthYear").alias("birth_year")
    )
)

In [0]:
dim_person.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{CATALOG}.gold.dim_person")

print(
    f"gold.dim_person: "
    f"{spark.table(f'{CATALOG}.gold.dim_person').count():,} linhas"
)

gold.dim_person: 1,279,523 linhas


> Criação da tabela de relacionamento Título × Elenco

Representa a participação dos atores e atrizes em filmes, séries e episódios, registrando os personagens interpretados e a ordem de aparição nos créditos.

In [0]:
silver_cast = spark.table(
    f"{CATALOG}.silver.title_cast"
)

bridge_title_cast = (
    silver_cast
    .select(
        F.col("tconst").alias("title_id"),
        F.col("nconst").alias("person_id"),
        F.col("characters"),
        F.col("ordering")
    )
)

In [0]:
bridge_title_cast.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(
        f"{CATALOG}.gold.bridge_title_cast"
    )

print(
    f"gold.bridge_title_cast: "
    f"{spark.table(f'{CATALOG}.gold.bridge_title_cast').count():,} linhas"
)

gold.bridge_title_cast: 10,028,177 linhas


In [0]:

silver_episode = spark.table(f"{CATALOG}.silver.title_episode")

dim_episode = silver_episode.select(
    F.col("tconst").alias("episode_title_id"),   # também existe como title_id em dim_title
    F.col("parentTconst").alias("series_title_id"),
    F.col("seasonNumber").alias("season_number"),
    F.col("episodeNumber").alias("episode_number"),
)

dim_episode.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.gold.dim_episode")
print(f"gold.dim_episode: {spark.table(f'{CATALOG}.gold.dim_episode').count():,} linhas")


gold.dim_episode: 7,337,797 linhas


> Criação da tabela fato de avaliações

Consolida as métricas de avaliação dos títulos, incluindo nota média e quantidade de votos recebidos no IMDb.

In [0]:
silver_ratings = spark.table(f"{CATALOG}.silver.title_ratings")
titles_in_scope = dim_title.select(F.col("title_id"))

fact_ratings = (
    silver_ratings
    .join(titles_in_scope, silver_ratings.tconst == titles_in_scope.title_id, "inner")
    .select(
        F.col("tconst").alias("title_id"),
        F.col("averageRating").alias("average_rating"),
        F.col("numVotes").alias("num_votes"),
    )
)

fact_ratings.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.gold.fact_ratings")
print(f"gold.fact_ratings: {spark.table(f'{CATALOG}.gold.fact_ratings').count():,} linhas")


gold.fact_ratings: 1,432,172 linhas


In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.vw_catalogo AS
SELECT
    t.title_id,
    t.title_type,
    t.primary_title,
    t.start_year,
    t.end_year,
    t.runtime_minutes,
    t.is_adult,
    r.average_rating,
    r.num_votes,
    array_join(collect_set(g.genre_name), ', ') AS generos
FROM {CATALOG}.gold.dim_title t
LEFT JOIN {CATALOG}.gold.fact_ratings r ON t.title_id = r.title_id
LEFT JOIN {CATALOG}.gold.bridge_title_genre btg ON t.title_id = btg.title_id
LEFT JOIN {CATALOG}.gold.dim_genre g ON btg.genre_id = g.genre_id
GROUP BY t.title_id, t.title_type, t.primary_title, t.start_year, t.end_year,
         t.runtime_minutes, t.is_adult, r.average_rating, r.num_votes
""")

display(spark.table(f"{CATALOG}.gold.vw_catalogo").limit(10))

title_id,title_type,primary_title,start_year,end_year,runtime_minutes,is_adult,average_rating,num_votes,generos
tt0138098,Filme,Eight in the Footstep of One,1964,null,90,false,6.8,25,"Family, Drama"
tt0477253,Filme,Aashiq Banaya Aapne: Love Takes Over,2005,null,115,false,4.4,2029,"Drama, Thriller, Romance"
tt0500969,Episódio,The Wild Bunch,2003,null,null,false,6.2,13,"Drama, Crime"
tt0544037,Episódio,A Good Death,2000,null,43,false,8.3,23,"Mystery, Drama, Crime"
tt0600204,Episódio,"God, Girls and Herman",1993,null,null,false,7.7,20,Comedy
tt0623247,Episódio,May I Ride with You,1994,null,60,false,6.6,49,"Action, Drama, Crime"
tt0644703,Episódio,"Howard, the Poet",1969,null,30,false,7.1,37,"Family, Comedy"
tt0646309,Episódio,The Back to School Syndrome,2000,null,20,false,6.2,56,"Animation, Action, Comedy"
tt0746973,Episódio,"Danny Kaye, Yves Saint-Laurent & Bette Davis",1965,null,30,false,9.5,149,"Family, Game-Show"
tt0763959,Episódio,Twenty Questions,1985,null,21,false,7.5,97,"Animation, Action, Adventure"


In [0]:
for t in ["dim_title", "dim_genre", "bridge_title_genre", "dim_person", "bridge_title_cast", "dim_episode", "fact_ratings"]:
    print(f"--- gold.{t} ---")
    spark.table(f"{CATALOG}.gold.{t}").printSchema()

--- gold.dim_title ---
root
 |-- title_id: string (nullable = true)
 |-- title_type: string (nullable = true)
 |-- title_type_original: string (nullable = true)
 |-- primary_title: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- start_year: integer (nullable = true)
 |-- end_year: integer (nullable = true)
 |-- runtime_minutes: integer (nullable = true)
 |-- is_adult: boolean (nullable = true)

--- gold.dim_genre ---
root
 |-- genre_id: integer (nullable = true)
 |-- genre_name: string (nullable = true)

--- gold.bridge_title_genre ---
root
 |-- title_id: string (nullable = true)
 |-- genre_id: integer (nullable = true)

--- gold.dim_person ---
root
 |-- person_id: string (nullable = true)
 |-- primary_name: string (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- death_year: integer (nullable = true)
 |-- professions: array (nullable = true)
 |    |-- element: string (containsNull = true)

--- gold.bridge_title_cast ---
root
 |-- title_id: s